In [1]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import tensorflow.keras.backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Add
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers, models, callbacks, backend as K
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf

/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

In [2]:
df_train = pd.read_csv('/kaggle/input/playground-series-s5e5/train.csv')
df_test  = pd.read_csv('/kaggle/input/playground-series-s5e5/test.csv')
df_sub = pd.read_csv('/kaggle/input/playground-series-s5e5/sample_submission.csv')

In [3]:
df_train.drop(columns=['id'], inplace=True)
df_test.drop(columns=['id'], inplace=True)


In [4]:
df_extra = pd.read_csv('/kaggle/input/calories-burning-dataset/exercise.csv')
df_cal = pd.read_csv('/kaggle/input/calories-burning-dataset/calories.csv')

In [5]:
df_extra.shape,df_train.shape

((15000, 8), (750000, 8))

In [6]:
df_extra.columns,df_train.columns

(Index(['User_ID', 'Gender', 'Age', 'Height', 'Weight', 'Duration',
        'Heart_Rate', 'Body_Temp'],
       dtype='object'),
 Index(['Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp',
        'Calories'],
       dtype='object'))

In [7]:
df_extra.drop(columns=['User_ID'], inplace=True)

In [8]:
df_extra.columns,df_train.columns

(Index(['Gender', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate',
        'Body_Temp'],
       dtype='object'),
 Index(['Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp',
        'Calories'],
       dtype='object'))

In [9]:
df_extra.head()

,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
0,male,68,190.0,94.0,29.0,105.0,40.8
1,female,20,166.0,60.0,14.0,94.0,40.3
2,male,69,179.0,79.0,5.0,88.0,38.7
3,female,34,179.0,71.0,13.0,100.0,40.5
4,female,27,154.0,58.0,10.0,81.0,39.8


In [10]:
df_extra.rename(columns={'Gender': 'Sex'}, inplace=True)
df_extra['Calories'] = df_cal['Calories'].values

In [11]:
df_extra.head()

,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,male,68,190.0,94.0,29.0,105.0,40.8,231.0
1,female,20,166.0,60.0,14.0,94.0,40.3,66.0
2,male,69,179.0,79.0,5.0,88.0,38.7,26.0
3,female,34,179.0,71.0,13.0,100.0,40.5,71.0
4,female,27,154.0,58.0,10.0,81.0,39.8,35.0


In [12]:
df_extra = df_extra[df_train.columns]  
df_train = pd.concat([df_train, df_extra], ignore_index=True)

In [13]:
df_train.shape

(765000, 8)

In [14]:
df_train.describe()

,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
count,765000.000000,765000.000000,765000.000000,765000.000000,765000.000000,765000.000000,765000.000000
mean,41.447255,174.693126,75.142162,15.423163,95.484672,40.036041,88.307424
std,15.213677,12.854173,14.004122,8.353421,9.452476,0.779863,62.396760
min,20.000000,123.000000,36.000000,1.000000,67.000000,37.100000,1.000000
25%,28.000000,164.000000,63.000000,8.000000,88.000000,39.600000,34.000000
50%,40.000000,174.000000,74.000000,15.000000,95.000000,40.300000,77.000000
75%,52.000000,185.000000,87.000000,23.000000,103.000000,40.700000,136.000000
max,79.000000,222.000000,132.000000,30.000000,128.000000,41.500000,314.000000


In [15]:
df_train.corr()

/tmp/ipykernel_20/299540020.py:1: FutureWarning: The default value of numeric_only in DataFrame.corr is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.
  df_train.corr()


,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
Age,1.000000,0.011884,0.074046,0.015623,0.016895,0.029871,0.145877
Height,0.011884,1.000000,0.957966,-0.029389,-0.012931,-0.033853,-0.003563
Weight,0.074046,0.957966,1.000000,-0.020450,-0.002242,-0.023128,0.016271
Duration,0.015623,-0.029389,-0.020450,1.000000,0.874879,0.903061,0.959819
Heart_Rate,0.016895,-0.012931,-0.002242,0.874879,1.000000,0.795482,0.908528
Body_Temp,0.029871,-0.033853,-0.023128,0.903061,0.795482,1.000000,0.828580
Calories,0.145877,-0.003563,0.016271,0.959819,0.908528,0.828580,1.000000


In [16]:
def create_predictor_features(df):   
    epsilon = 1e-5
    df['BMI'] = df['Weight'] / ((df['Height'] / 100) ** 2)
    df['Duration_HR'] = df['Duration'] * df['Heart_Rate']
    df['Duration_Temp'] = df['Duration'] * df['Body_Temp']
    df['HR_Temp'] = df['Heart_Rate'] * df['Body_Temp']
    df['Combined_Effort'] = df['Duration'] * df['Heart_Rate'] * df['Body_Temp']
    df['Weight_to_Height'] = df['Weight'] / (df['Height'] + epsilon)
    df['Duration_per_Age'] = df['Duration'] / (df['Age'] + epsilon)
    df['HR_per_Age'] = df['Heart_Rate'] / (df['Age'] + epsilon)
    df['Temp_per_Age'] = df['Body_Temp'] / (df['Age'] + epsilon)
    for col in ['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp']:
        df[f'{col}_zscore'] = (df[col] - df[col].mean()) / df[col].std()
    df['Duration_squared'] = df['Duration'] ** 2
    df['HR_squared'] = df['Heart_Rate'] ** 2
    df['Temp_squared'] = df['Body_Temp'] ** 2
    for col in ['Duration', 'Heart_Rate', 'Body_Temp', 'Weight']:
        df[f'{col}_log'] = np.log1p(df[col])
    df['Age_Group'] = pd.cut(df['Age'], bins=[19, 30, 45, 60, 80], labels=['20-30', '31-45', '46-60', '61-80'])
    df['BMI_Category'] = pd.cut(df['BMI'], bins=[0, 18.5, 24.9, 29.9, 40], labels=['Underweight', 'Normal', 'Overweight', 'Obese'])
    return df

In [17]:

#df_train = create_predictor_features(df_train)
#df_test = create_predictor_features(df_test)


In [18]:
numerical_features = ['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp']

def add_feature_cross_terms(df, numerical_features):
    df_new = df.copy()
    for i in range(len(numerical_features)):
        for j in range(i + 1, len(numerical_features)):
            feature1 = numerical_features[i]
            feature2 = numerical_features[j]
            cross_term_name = f"{feature1}_x_{feature2}"
            df_new[cross_term_name] = df_new[feature1] * df_new[feature2]
    return df_new

df_train = add_feature_cross_terms(df_train, numerical_features)
df_test = add_feature_cross_terms(df_test, numerical_features)


In [19]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

le = LabelEncoder()
df_train['Sex'] = le.fit_transform(df_train['Sex'])
df_test['Sex'] = le.transform(df_test['Sex'])
#le_age = LabelEncoder()
#le_bmi = LabelEncoder()
#df_train['Age_Group'] = le_age.fit_transform(df_train['Age_Group'].astype(str))
#df_train['BMI_Category'] = le_bmi.fit_transform(df_train['BMI_Category'].astype(str))
#df_test['Age_Group'] = le_age.transform(df_test['Age_Group'].astype(str))
#df_test['BMI_Category'] = le_bmi.transform(df_test['BMI_Category'].astype(str))

In [20]:
df_train.columns,df_test.columns

(Index(['Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp',
        'Calories', 'Age_x_Height', 'Age_x_Weight', 'Age_x_Duration',
        'Age_x_Heart_Rate', 'Age_x_Body_Temp', 'Height_x_Weight',
        'Height_x_Duration', 'Height_x_Heart_Rate', 'Height_x_Body_Temp',
        'Weight_x_Duration', 'Weight_x_Heart_Rate', 'Weight_x_Body_Temp',
        'Duration_x_Heart_Rate', 'Duration_x_Body_Temp',
        'Heart_Rate_x_Body_Temp'],
       dtype='object'),
 Index(['Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp',
        'Age_x_Height', 'Age_x_Weight', 'Age_x_Duration', 'Age_x_Heart_Rate',
        'Age_x_Body_Temp', 'Height_x_Weight', 'Height_x_Duration',
        'Height_x_Heart_Rate', 'Height_x_Body_Temp', 'Weight_x_Duration',
        'Weight_x_Heart_Rate', 'Weight_x_Body_Temp', 'Duration_x_Heart_Rate',
        'Duration_x_Body_Temp', 'Heart_Rate_x_Body_Temp'],
       dtype='object'))

In [21]:
#df_train['BMI_Category']

In [22]:
def root_mean_squared_error(y_true, y_pred):
    return tf.sqrt(tf.reduce_mean(tf.square(y_pred - y_true)))

In [23]:
def swish(x):
    return x * tf.keras.backend.sigmoid(x)

In [24]:
def build_swish_mlp(input_shape):
    inputs = Input(shape=input_shape)
    x = Dense(32, activation=swish)(inputs)
    x = Dense(64, activation=swish)(x)
    x = Dense(32, activation=swish)(x)
    x = Dense(1)(x)
    model = Model(inputs, x)
    model.compile(optimizer=Adam(1e-3), loss='mse', metrics=[root_mean_squared_error])
    return model

X_train = df_train.drop('Calories', axis=1).values.astype(np.float32)
y_train = df_train['Calories'].values.astype(np.float32)
X_test = df_test.values.astype(np.float32)

y_train_log = np.log1p(y_train)


scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


model = build_swish_mlp(input_shape=(X_train_scaled.shape[1],))
model.fit(X_train_scaled, y_train_log, epochs=20, batch_size=64, verbose=1) 

Epoch 1/20
11954/11954 [==============================] - 30s 2ms/step - loss: 0.0469 - root_mean_squared_error: 0.0920
Epoch 2/20
11954/11954 [==============================] - 26s 2ms/step - loss: 0.0041 - root_mean_squared_error: 0.0604
Epoch 3/20
11954/11954 [==============================] - 25s 2ms/step - loss: 0.0040 - root_mean_squared_error: 0.0596
Epoch 4/20
11954/11954 [==============================] - 24s 2ms/step - loss: 0.0040 - root_mean_squared_error: 0.0593
Epoch 5/20
11954/11954 [==============================] - 24s 2ms/step - loss: 0.0039 - root_mean_squared_error: 0.0589
Epoch 6/20
11954/11954 [==============================] - 24s 2ms/step - loss: 0.0039 - root_mean_squared_error: 0.0588
Epoch 7/20
11954/11954 [==============================] - 24s 2ms/step - loss: 0.0039 - root_mean_squared_error: 0.0586
Epoch 8/20
11954/11954 [==============================] - 24s 2ms/step - loss: 0.0039 - root_mean_squared_error: 0.0585
Epoch 9/20
11954/11954 [================

In [25]:
y_test_pred = model.predict(X_test_scaled)
final_pred = np.expm1(y_test_pred).flatten()
final_pred = np.clip(final_pred, y_train.min(), y_train.max())  

7813/7813 [==============================] - 11s 1ms/step


In [26]:
df_sub['Calories'] = final_pred

In [27]:
df_sub.to_csv('submission.csv', index=False)

In [28]:
df_sub

,id,Calories
0,750000,27.384996
1,750001,108.401642
2,750002,87.269096
3,750003,125.779358
4,750004,76.016335
...,...,...
249995,999995,26.043108
249996,999996,9.456956
249997,999997,73.631660
249998,999998,169.045563
